# Hyperparameter Tuning & Model Selection

Light, validation-driven tuning of the XGBoost delay classifier. We search a
small grid of tree-shape and learning-rate settings, select the configuration
with the best validation PR-AUC, refit it as the final model, and persist the
artifacts needed for serving.

In [1]:
from pyprojroot import here
print("Project root:", here())

Project root: C:\Users\hanis\Flight\flight-delay-prediction


## Imports

`fit_encoders` / `transform_features` come from our `features` module — the
learn/apply split that guarantees identical feature logic at train, validation,
and serving time. `average_precision_score` (PR-AUC) is our selection metric:
the target is imbalanced (~17% delayed), so accuracy and ROC-AUC would flatter
the model. PR-AUC reflects performance on the minority (delayed) class.

In [2]:
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import average_precision_score

from xgboost import XGBClassifier

from features import (fit_encoders,transform_features)

## Load training and validation splits

The splits are **chronological** — train = Aug–Oct, validation = November. This
is a temporal split, not a random shuffle, so no future flights leak into
training. The December test set is not loaded here.

In [3]:
train_df = pd.read_parquet(here("data/processed/train.parquet"))
val_df = pd.read_parquet(here("data/processed/val.parquet"))

## Fit encoders on train, apply to train and validation

`fit_encoders` learns the historical delay-rate encodings (airline, origin,
dest) from the **training set only**. `transform_features` then applies those
learned mappings to train and validation alike. Because the encoders are fit on
train and merely *applied* to validation, the validation set never sees its own
target — the leak-free guarantee holds end to end.

In [4]:
encoders = fit_encoders(train_df)

X_train = transform_features(
    train_df,
    encoders
)

X_val = transform_features(
    val_df,
    encoders
)

y_train = train_df["DepDel15"]
y_val = val_df["DepDel15"]

X_train.head()

,DayOfWeek,dep_hour,Distance,airline_delay_rate,origin_delay_rate,dest_delay_rate
0,2,13,2133.0,0.150539,0.183525,0.167577
1,2,19,455.0,0.150539,0.190725,0.162454
2,2,9,1041.0,0.150539,0.196701,0.194043
3,2,13,888.0,0.150539,0.196701,0.156885
4,2,12,1846.0,0.150539,0.196701,0.179131


## Handling class imbalance

With only ~17% of flights delayed, we set `scale_pos_weight = negatives /
positives ≈ 4.8` so XGBoost weights the minority (delayed) class during
training instead of optimising for the on-time majority. Note: this inflates
predicted probabilities, which we account for when choosing a decision
threshold later.

In [5]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()

scale_pos_weight = neg / pos

print(f"Negatives: {neg}")
print(f"Positives: {pos}")
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

Negatives: 1482467
Positives: 308612
scale_pos_weight = 4.80


## Light grid search with early stopping

We sweep a small grid — `max_depth ∈ {4, 6, 8}` × `learning_rate ∈ {0.05, 0.1}`
— six configurations. For each:

- `n_estimators` is capped at 2000, but **early stopping** on the validation set
  (`aucpr`, patience 50) selects the actual tree count — we never hand-tune it.
- `subsample` and `colsample_bytree = 0.8` add regularisation.
- Each fit is scored by **validation PR-AUC**.

Selection uses the fixed temporal validation set — **not** shuffled k-fold
cross-validation, which would mix November flights into training and break the
temporal integrity of the split.

Baseline: a no-skill classifier scores PR-AUC ≈ the positive rate (~0.14 on the
November validation set). That is the floor every configuration must beat.

In [6]:
results = []

for depth in [4, 6, 8]:
    for lr in [0.05, 0.1]:

        model = XGBClassifier(
            n_estimators=2000,
            max_depth=depth,
            learning_rate=lr,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="binary:logistic",
            eval_metric="aucpr",
            early_stopping_rounds=50,
            random_state=42
        )

        model.fit(
            X_train,
            y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        probs = model.predict_proba(X_val)[:, 1]

        pr_auc = average_precision_score(
            y_val,
            probs
        )

        results.append({
            "max_depth": depth,
            "learning_rate": lr,
            "pr_auc": pr_auc,
            "best_iteration": model.best_iteration
        })

        print(
            f"depth={depth}, "
            f"lr={lr}, "
            f"PR-AUC={pr_auc:.4f}, "
            f"trees={model.best_iteration}"
        )

depth=4, lr=0.05, PR-AUC=0.2231, trees=1350
depth=4, lr=0.1, PR-AUC=0.2232, trees=798
depth=6, lr=0.05, PR-AUC=0.2255, trees=1144
depth=6, lr=0.1, PR-AUC=0.2259, trees=699
depth=8, lr=0.05, PR-AUC=0.2258, trees=558
depth=8, lr=0.1, PR-AUC=0.2257, trees=325


## Tuning results

The six configurations, ranked by validation PR-AUC:

| max_depth | learning_rate | PR-AUC | trees (early-stopped) |
|-----------|---------------|--------|------------------------|
| 4 | 0.05 | 0.2231 | 1350 |
| 4 | 0.10 | 0.2232 | 798 |
| 6 | 0.05 | 0.2255 | 1144 |
| 6 | 0.10 | 0.2259 | 699 |
| 8 | 0.05 | 0.2258 | 558 |
| 8 | 0.10 | 0.2257 | 325 |

**Best configuration:** `max_depth = 8`, `learning_rate = 0.05`, PR-AUC = 0.2259
(vs a no-skill baseline of ~0.14).

**Observations**

- All six configurations cluster within a very narrow band (0.2226–0.2259, a
  spread of just 0.003). The model is largely insensitive to these
  hyperparameters, which indicates performance is bounded by the **feature
  signal**, not by tuning — additional gains will come from richer features, not
  a finer grid.
- Greater tree depth helps marginally (depth 4 ≈ 0.223 → depth 6/8 ≈ 0.225).
  Learning rate has almost no effect on the final score; it mainly

In [7]:
results_df = pd.DataFrame(results)

results_df.sort_values(
    "pr_auc",
    ascending=False
)

,max_depth,learning_rate,pr_auc,best_iteration
3,6,0.10,0.225919,699
4,8,0.05,0.225804,558
5,8,0.10,0.225680,325
2,6,0.05,0.225452,1144
1,4,0.10,0.223191,798
0,4,0.05,0.223051,1350


## Select the best configuration

Take the top configuration by validation PR-AUC and extract its `max_depth` and
`learning_rate` for the final model.

In [8]:
best_row = results_df.sort_values(
    "pr_auc",
    ascending=False
).iloc[0]

best_depth = int(best_row["max_depth"])
best_lr = float(best_row["learning_rate"])

## Refit the final model

Retrain a single clean model on the selected hyperparameters, again with early
stopping on validation. Trained on the **training split only** (not train+val),
which keeps the model consistent with the encoders — also fit on train alone.

In [9]:
final_model = XGBClassifier(
    n_estimators=2000,
    max_depth=best_depth,
    learning_rate=best_lr,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="aucpr",
    early_stopping_rounds=50,
    random_state=42
)

final_model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    verbose=False
)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8
,device,None
,early_stopping_rounds,50
,enable_categorical,True
,eval_metric,'aucpr'


## Persist artifacts

Saved in the same run that produced the model, so the mappings are never
reconstructed later:

- `xgb_final.pkl` — the trained model
- `encoders.pkl` — the train-only delay-rate mappings (the "fit" output)
- `feature_columns.pkl` — the exact feature order the model expects
- `tuning_results.csv` — the search log, for reproducibility

In [10]:
joblib.dump(
    final_model,
    here("models/xgb_final.pkl")
)

joblib.dump(
    encoders,
    here("models/encoders.pkl")
)

joblib.dump(
    list(X_train.columns),
    here("models/feature_columns.pkl")
)

results_df.to_csv(
    here("models/tuning_results.csv"),
    index=False
)

## Final validation score

Validation PR-AUC of the selected model: **0.2259**, against a no-skill baseline
of ~0.14. The test set remains untouched, reserved for the final evaluation.

In [11]:
val_probs = final_model.predict_proba(X_val)[:, 1]

val_pr_auc = average_precision_score(
    y_val,
    val_probs
)

print(f"Validation PR-AUC: {val_pr_auc:.4f}")

Validation PR-AUC: 0.2259
